In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import hdbscan
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist, squareform
from sklearn.covariance import MinCovDet
import os
import time
from sklearn.utils.extmath import fast_logdet

In [4]:
import numpy as np
import pandas as pd
import hdbscan
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd 
import os

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA  # <-- Missing import
import matplotlib.pyplot as plt
#data load
raw_counts = pd.read_csv("Final data_.csv", index_col=0)

In [5]:
raw_counts.columns = raw_counts.columns.str.replace(r'^(CONTROL|TREATMENT).*', r'\1', regex=True)
print("\nCleaned columns:")
print(raw_counts.columns)


Cleaned columns:
Index(['CONTROL', 'CONTROL', 'CONTROL', 'TREATMENT', 'TREATMENT', 'TREATMENT',
       'CONTROL', 'CONTROL', 'CONTROL', 'TREATMENT',
       ...
       'CONTROL', 'CONTROL', 'TREATMENT', 'TREATMENT', 'CONTROL', 'CONTROL',
       'TREATMENT', 'TREATMENT', 'CONTROL', 'CONTROL'],
      dtype='object', length=202)


In [8]:
raw_counts.describe()

,CONTROL,CONTROL,CONTROL,TREATMENT,TREATMENT,TREATMENT,CONTROL,CONTROL,CONTROL,TREATMENT,...,CONTROL,CONTROL,TREATMENT,TREATMENT,CONTROL,CONTROL,TREATMENT,TREATMENT,CONTROL,CONTROL
count,28581.000000,28581.000000,28581.000000,28581.000000,28581.000000,28581.000000,28581.000000,28581.000000,28581.000000,28581.000000,...,28581.000000,2.858100e+04,28581.000000,28581.000000,28581.000000,28581.000000,28581.000000,28581.000000,28581.00000,28581.000000
mean,337.338861,258.200518,354.676218,310.343165,286.769287,243.725832,244.345719,277.621392,113.155313,143.355726,...,955.742521,2.310960e+03,426.131731,490.452783,358.515167,395.439348,426.872328,376.962702,451.02904,390.957979
std,2130.644672,2007.501541,1656.052777,3108.427121,2455.910174,2112.309010,1834.986168,2245.295346,1079.189744,1826.177229,...,8928.896065,2.418052e+04,1871.779305,1840.755694,1415.776448,1415.100989,1631.919288,1548.640672,2122.24816,1668.899471
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,1.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
50%,31.000000,19.000000,40.000000,33.000000,30.000000,26.000000,22.000000,36.000000,10.000000,13.000000,...,75.000000,1.890000e+02,63.000000,83.000000,57.000000,57.000000,72.000000,61.000000,48.00000,62.000000
75%,263.000000,180.000000,294.000000,223.000000,215.000000,182.000000,183.000000,234.000000,80.000000,96.000000,...,514.000000,1.202000e+03,346.000000,432.000000,308.000000,323.000000,386.000000,331.000000,314.00000,319.000000
max,255372.000000,283232.000000,155208.000000,448765.000000,349440.000000,291930.000000,259124.000000,338622.000000,160190.000000,280931.000000,...,785636.000000,1.940297e+06,88967.000000,81162.000000,66518.000000,80289.000000,81983.000000,82531.000000,91748.00000,93326.000000


In [9]:
raw_counts.head()
raw_counts.info

<bound method DataFrame.info of               CONTROL  CONTROL  CONTROL  TREATMENT  TREATMENT  TREATMENT  \
GeneID                                                                     
LOC101497325        3        1        4          2          0         10   
LOC101488339       66       34       95         78         84         85   
LOC101488545       93       59       90         60         45         64   
LOC101497647      154      136      153        156        148         65   
LOC101489113      729      534      760        502        604        441   
...               ...      ...      ...        ...        ...        ...   
CiarC_p072          0        0        0          0          0          0   
CiarC_p073          0        0        0          0          0          0   
CiarC_t029          0        0        0          0          0          0   
CiarC_p074          0        0        0          0          0          0   
CiarC_p075          0        0        0          0      

In [10]:
filtered_df = raw_counts[raw_counts.ne(0).any(axis=1)]

In [14]:

filtered_df.describe()

,CONTROL,CONTROL,CONTROL,TREATMENT,TREATMENT,TREATMENT,CONTROL,CONTROL,CONTROL,TREATMENT,...,CONTROL,CONTROL,TREATMENT,TREATMENT,CONTROL,CONTROL,TREATMENT,TREATMENT,CONTROL,CONTROL
count,27038.000000,27038.000000,27038.000000,27038.000000,27038.000000,27038.000000,27038.000000,27038.000000,27038.000000,27038.000000,...,27038.000000,2.703800e+04,27038.000000,27038.000000,27038.000000,27038.000000,27038.000000,27038.000000,27038.000000,27038.000000
mean,356.590058,272.935461,374.916821,328.053776,303.134588,257.634736,258.289999,293.464642,119.612841,151.536726,...,1010.284673,2.442842e+03,450.450144,518.441860,378.974850,418.006213,451.233005,398.475146,476.768289,413.269103
std,2189.031532,2063.016245,1700.422568,3194.986213,2524.034620,2170.922220,1885.666161,2307.468570,1109.209001,1877.233970,...,9177.145058,2.485445e+04,1921.601146,1888.715175,1452.949159,1451.675082,1674.560907,1589.523700,2179.152261,1713.171695
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,...,1.000000,3.000000e+00,1.000000,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
50%,43.000000,27.000000,55.000000,45.000000,42.000000,36.000000,32.000000,48.000000,14.000000,18.000000,...,101.000000,2.490000e+02,79.000000,102.000000,73.000000,72.000000,93.000000,79.000000,64.000000,80.000000
75%,285.000000,199.000000,319.000000,243.000000,232.000000,196.000000,199.000000,253.000000,87.000000,104.000000,...,559.000000,1.310750e+03,375.000000,468.000000,333.000000,349.000000,416.000000,361.000000,342.000000,342.000000
max,255372.000000,283232.000000,155208.000000,448765.000000,349440.000000,291930.000000,259124.000000,338622.000000,160190.000000,280931.000000,...,785636.000000,1.940297e+06,88967.000000,81162.000000,66518.000000,80289.000000,81983.000000,82531.000000,91748.000000,93326.000000


Imports and directory creations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import time
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.covariance import MinCovDet
import hdbscan
from scipy import linalg

def fast_logdet(matrix):
    """Compute log determinant efficiently"""
    return 2 * np.sum(np.log(np.diag(linalg.cholesky(matrix))))

# Create output directory
output_dir = "final_results/distance_metric_comparison_again_for_FOR_3k_GENES/check"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
DATA normalization and extraction of top variable genes

In [15]:

# 1. Data preparation
normalized_data = np.log2(filtered_df + 1)  # Log2 transform with pseudocount

# 2. Select top variable genes
gene_variability = normalized_data.std(axis=1)
top_genes = gene_variability.sort_values(ascending=False).head(3000).index
top_variable_data = normalized_data.loc[top_genes]

print(f"Top variable genes shape: {top_variable_data.shape}")

top_variable_data = pd.DataFrame(
    StandardScaler().fit_transform(top_variable_data),
    index=top_genes,  # CORRECTED: Changed from top_genes_idx to top_genes
    columns=top_variable_data.columns
)

Top variable genes shape: (3000, 202)


In [16]:
class RobustMahalanobisHDBSCAN:
    def __init__(self, min_cluster_size=5, min_samples=None):
        self.min_cluster_size = min_cluster_size
        self.min_samples = min_samples
        
    def _robust_mahalanobis(self, X):
        """Calculate pairwise Mahalanobis distances with regularization"""
        try:
            # Robust covariance estimation with increased support_fraction
            cov_estimator = MinCovDet(support_fraction=0.9).fit(X)
            cov = cov_estimator.covariance_
            
            # Regularization to ensure invertibility
            reg = 1e-6 * np.trace(cov)/cov.shape[0] * np.eye(cov.shape[0])
            cov_reg = cov + reg
            
            # Check condition number
            if fast_logdet(cov_reg) < -50:  # If nearly singular
                raise ValueError("Covariance matrix is nearly singular")
                
            cov_inv = np.linalg.inv(cov_reg)
            
            # Calculate distances using matrix operations
            diff = X[:, np.newaxis, :] - X[np.newaxis, :, :]
            dists = np.sqrt(np.einsum('...i,ij,...j->...', diff, cov_inv, diff))
            np.fill_diagonal(dists, 0)  # Ensure diagonal is exactly 0
            return dists
            
        except Exception as e:
            print(f"Mahalanobis calculation failed: {str(e)}")
            return None

    def fit(self, X):
        dist_matrix = self._robust_mahalanobis(X)
        if dist_matrix is None:
            # Fallback to Euclidean if Mahalanobis fails
            print("Using Euclidean as fallback")
            self.clusterer = hdbscan.HDBSCAN(
                min_cluster_size=self.min_cluster_size,
                min_samples=self.min_samples
            )
        else:
            self.clusterer = hdbscan.HDBSCAN(
                min_cluster_size=self.min_cluster_size,
                min_samples=self.min_samples,
                metric='precomputed'
            )
            
        self.labels_ = self.clusterer.fit_predict(dist_matrix if dist_matrix is not None else X)
        return self

# ----------------------------
# Metric Comparison
# ----------------------------
metrics = [
    ('euclidean', hdbscan.HDBSCAN(metric='euclidean')),
    ('manhattan', hdbscan.HDBSCAN(metric='manhattan')),
    ('correlation', hdbscan.HDBSCAN(metric='correlation')),
    ('mahalanobis', RobustMahalanobisHDBSCAN())
]

results = []
for name, clusterer in metrics:
    print(f"\n{'='*40}\nClustering with: {name.upper()}\n{'='*40}")
    
    try:
        start_time = time.time()
        
        if name == 'mahalanobis':
            print("Computing Mahalanobis distances...")
            clusterer.fit(top_variable_data.values)
            clusters = clusterer.labels_
        else:
            clusters = clusterer.fit_predict(top_variable_data)
            
        # Analysis
        unique_clusters = np.unique(clusters)
        n_clusters = len(unique_clusters) - 1  # Exclude noise (-1)
        noise_ratio = np.mean(clusters == -1)
        cluster_sizes = [np.sum(clusters == c) for c in unique_clusters if c != -1]
        
        results.append({
            'metric': name,
            'n_clusters': n_clusters,
            'noise_ratio': noise_ratio,
            'clusters': clusters,
            'cluster_sizes': cluster_sizes,
            'time': time.time() - start_time
        })
        
        print(f"Number of clusters: {n_clusters}")
        print(f"Noise ratio: {noise_ratio:.1%}")
        print(f"Cluster sizes: {sorted(cluster_sizes, reverse=True)}")
        print(f"Time elapsed: {results[-1]['time']:.2f} seconds")
        
    except Exception as e:
        print(f"Clustering failed: {str(e)}")
        results.append({
            'metric': name,
            'error': str(e),
            'clusters': None
        })

# ----------------------------
# Visualization
# ----------------------------
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()

pca = PCA(n_components=2).fit_transform(top_variable_data)

for i, result in enumerate(results):
    ax = axes[i]
    
    if result['clusters'] is None:
        ax.text(0.5, 0.5, f"Failed:\n{result.get('error','')}", 
                ha='center', va='center', fontsize=12)
        ax.set_title(f"{result['metric'].upper()} - FAILED", fontsize=14)
        continue
        
    sc = ax.scatter(
        pca[:, 0], pca[:, 1],
        c=result['clusters'],
        cmap='tab20',
        alpha=0.7,
        s=15
    )
    
    title = (
        f"{result['metric'].upper()}\n"
        f"Clusters: {result['n_clusters']} | "
        f"Noise: {result['noise_ratio']:.1%}\n"
        f"Time: {result['time']:.2f}s"
    )
    ax.set_title(title, fontsize=12)
    
    # Add colorbar for cluster IDs
    plt.colorbar(sc, ax=ax, label='Cluster ID')

plt.tight_layout()
plt.savefig(f"{output_dir}/metric_comparison.png", dpi=300, bbox_inches='tight')
plt.close()

# ----------------------------
# Cluster Size Distribution Plot
# ----------------------------
plt.figure(figsize=(12, 6))
for result in results:
    if result['clusters'] is not None and result['cluster_sizes']:
        sizes = result['cluster_sizes']
        plt.plot(sorted(sizes, reverse=True), 'o-', label=result['metric'].upper())

plt.xlabel("Cluster Rank", fontsize=12)
plt.ylabel("Number of Genes", fontsize=12)
plt.title("Cluster Size Distribution by Distance Metric", fontsize=14)
plt.yscale('log')
plt.legend()
plt.grid(True, which="both", ls="--")
plt.savefig(f"{output_dir}/cluster_size_distribution.png", dpi=300, bbox_inches='tight')
plt.close()

# ----------------------------
# Save Results
# ----------------------------
for result in results:
    if result['clusters'] is not None:
        pd.DataFrame({
            'gene': top_variable_data.index,
            'cluster': result['clusters']
        }).to_csv(f"{output_dir}/clusters_{result['metric']}.csv", index=False)

# Save summary statistics
summary_data = []
for r in results:
    summary_data.append({
        'metric': r['metric'],
        'n_clusters': r.get('n_clusters', np.nan),
        'noise_ratio': r.get('noise_ratio', np.nan),
        'largest_cluster': max(r['cluster_sizes']) if 'cluster_sizes' in r and r['cluster_sizes'] else np.nan,
        'time_seconds': r.get('time', np.nan),
        'status': 'SUCCESS' if r['clusters'] is not None else 'FAILED'
    })

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(f"{output_dir}/summary_stats.csv", index=False)

print(f"\nAnalysis complete. Results saved to: {output_dir}")
print("\nSummary Statistics:")
print(summary_df.to_string(index=False))


Clustering with: EUCLIDEAN
Number of clusters: 5
Noise ratio: 8.4%
Cluster sizes: [2717, 11, 8, 6, 5]
Time elapsed: 4.39 seconds

Clustering with: MANHATTAN
Number of clusters: 12
Noise ratio: 44.2%
Cluster sizes: [1266, 186, 93, 35, 27, 14, 14, 10, 8, 7, 7, 6]
Time elapsed: 4.70 seconds

Clustering with: CORRELATION
Number of clusters: 10
Noise ratio: 33.6%
Cluster sizes: [1085, 664, 126, 28, 25, 20, 14, 13, 10, 7]
Time elapsed: 1.20 seconds

Clustering with: MAHALANOBIS
Computing Mahalanobis distances...


C:\Users\HP\AppData\Roaming\Python\Python311\site-packages\sklearn\covariance\_robust_covariance.py:753: UserWarning: The covariance matrix associated to your dataset is not full rank
  warnings.warn(
C:\Users\HP\AppData\Roaming\Python\Python311\site-packages\sklearn\covariance\_robust_covariance.py:185: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-894.899231369978338 > -925.210490210406874). You may want to try with a higher value of support_fraction (current value: 0.900).
  warnings.warn(
C:\Users\HP\AppData\Roaming\Python\Python311\site-packages\sklearn\covariance\_robust_covariance.py:185: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-925.052839275076508 > -927.784331163067236). You may want to try with a higher value of support_fraction (current value: 0.900).
  warnings.warn(
C:\Users\HP\AppData\Roaming\Python\Python311\site-packages\sklearn\covariance\_robust_covariance.py:1

Mahalanobis calculation failed: Covariance matrix is nearly singular
Using Euclidean as fallback
Number of clusters: 5
Noise ratio: 8.4%
Cluster sizes: [2717, 11, 8, 6, 5]
Time elapsed: 24.21 seconds

Analysis complete. Results saved to: final_results/distance_metric_comparison_again_for5k_check/check

Summary Statistics:
     metric  n_clusters  noise_ratio  largest_cluster  time_seconds  status
  euclidean           5     0.084333             2717      4.391114 SUCCESS
  manhattan          12     0.442333             1266      4.698429 SUCCESS
correlation          10     0.336000             1085      1.200722 SUCCESS
mahalanobis           5     0.084333             2717     24.208442 SUCCESS


For 5000 top variable genes

Load the data and prepare the directories

In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import time
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.covariance import MinCovDet
import hdbscan
from scipy import linalg

def fast_logdet(matrix):
    """Compute log determinant efficiently"""
    return 2 * np.sum(np.log(np.diag(linalg.cholesky(matrix))))

# Create output directory
output_dir = "final_results/distance_metric_comparison_again_for_5k_GENES/check"
os.makedirs(output_dir, exist_ok=True)


Data normalization and extraction of 5K variable genes

In [18]:

# 1. Data preparation
normalized_data = np.log2(filtered_df + 1)  # Log2 transform with pseudocount

# 2. Select top variable genes
gene_variability = normalized_data.std(axis=1)
top_genes = gene_variability.sort_values(ascending=False).head(5000).index
top_variable_data = normalized_data.loc[top_genes]

print(f"Top variable genes shape: {top_variable_data.shape}")

top_variable_data = pd.DataFrame(
    StandardScaler().fit_transform(top_variable_data),
    index=top_genes,  # CORRECTED: Changed from top_genes_idx to top_genes
    columns=top_variable_data.columns
)


Top variable genes shape: (5000, 202)


In [19]:
class RobustMahalanobisHDBSCAN:
    def __init__(self, min_cluster_size=5, min_samples=None):
        self.min_cluster_size = min_cluster_size
        self.min_samples = min_samples
        
    def _robust_mahalanobis(self, X):
        """Calculate pairwise Mahalanobis distances with regularization"""
        try:
            # Robust covariance estimation with increased support_fraction
            cov_estimator = MinCovDet(support_fraction=0.9).fit(X)
            cov = cov_estimator.covariance_
            
            # Regularization to ensure invertibility
            reg = 1e-6 * np.trace(cov)/cov.shape[0] * np.eye(cov.shape[0])
            cov_reg = cov + reg
            
            # Check condition number
            if fast_logdet(cov_reg) < -50:  # If nearly singular
                raise ValueError("Covariance matrix is nearly singular")
                
            cov_inv = np.linalg.inv(cov_reg)
            
            # Calculate distances using matrix operations
            diff = X[:, np.newaxis, :] - X[np.newaxis, :, :]
            dists = np.sqrt(np.einsum('...i,ij,...j->...', diff, cov_inv, diff))
            np.fill_diagonal(dists, 0)  # Ensure diagonal is exactly 0
            return dists
            
        except Exception as e:
            print(f"Mahalanobis calculation failed: {str(e)}")
            return None

    def fit(self, X):
        dist_matrix = self._robust_mahalanobis(X)
        if dist_matrix is None:
            # Fallback to Euclidean if Mahalanobis fails
            print("Using Euclidean as fallback")
            self.clusterer = hdbscan.HDBSCAN(
                min_cluster_size=self.min_cluster_size,
                min_samples=self.min_samples
            )
        else:
            self.clusterer = hdbscan.HDBSCAN(
                min_cluster_size=self.min_cluster_size,
                min_samples=self.min_samples,
                metric='precomputed'
            )
            
        self.labels_ = self.clusterer.fit_predict(dist_matrix if dist_matrix is not None else X)
        return self

# ----------------------------
# Metric Comparison
# ----------------------------
metrics = [
    ('euclidean', hdbscan.HDBSCAN(metric='euclidean')),
    ('manhattan', hdbscan.HDBSCAN(metric='manhattan')),
    ('correlation', hdbscan.HDBSCAN(metric='correlation')),
    ('mahalanobis', RobustMahalanobisHDBSCAN())
]

results = []
for name, clusterer in metrics:
    print(f"\n{'='*40}\nClustering with: {name.upper()}\n{'='*40}")
    
    try:
        start_time = time.time()
        
        if name == 'mahalanobis':
            print("Computing Mahalanobis distances...")
            clusterer.fit(top_variable_data.values)
            clusters = clusterer.labels_
        else:
            clusters = clusterer.fit_predict(top_variable_data)
            
        # Analysis
        unique_clusters = np.unique(clusters)
        n_clusters = len(unique_clusters) - 1  # Exclude noise (-1)
        noise_ratio = np.mean(clusters == -1)
        cluster_sizes = [np.sum(clusters == c) for c in unique_clusters if c != -1]
        
        results.append({
            'metric': name,
            'n_clusters': n_clusters,
            'noise_ratio': noise_ratio,
            'clusters': clusters,
            'cluster_sizes': cluster_sizes,
            'time': time.time() - start_time
        })
        
        print(f"Number of clusters: {n_clusters}")
        print(f"Noise ratio: {noise_ratio:.1%}")
        print(f"Cluster sizes: {sorted(cluster_sizes, reverse=True)}")
        print(f"Time elapsed: {results[-1]['time']:.2f} seconds")
        
    except Exception as e:
        print(f"Clustering failed: {str(e)}")
        results.append({
            'metric': name,
            'error': str(e),
            'clusters': None
        })

# ----------------------------
# Visualization
# ----------------------------
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()

pca = PCA(n_components=2).fit_transform(top_variable_data)

for i, result in enumerate(results):
    ax = axes[i]
    
    if result['clusters'] is None:
        ax.text(0.5, 0.5, f"Failed:\n{result.get('error','')}", 
                ha='center', va='center', fontsize=12)
        ax.set_title(f"{result['metric'].upper()} - FAILED", fontsize=14)
        continue
        
    sc = ax.scatter(
        pca[:, 0], pca[:, 1],
        c=result['clusters'],
        cmap='tab20',
        alpha=0.7,
        s=15
    )
    
    title = (
        f"{result['metric'].upper()}\n"
        f"Clusters: {result['n_clusters']} | "
        f"Noise: {result['noise_ratio']:.1%}\n"
        f"Time: {result['time']:.2f}s"
    )
    ax.set_title(title, fontsize=12)
    
    # Add colorbar for cluster IDs
    plt.colorbar(sc, ax=ax, label='Cluster ID')

plt.tight_layout()
plt.savefig(f"{output_dir}/metric_comparison.png", dpi=300, bbox_inches='tight')
plt.close()

# ----------------------------
# Cluster Size Distribution Plot
# ----------------------------
plt.figure(figsize=(12, 6))
for result in results:
    if result['clusters'] is not None and result['cluster_sizes']:
        sizes = result['cluster_sizes']
        plt.plot(sorted(sizes, reverse=True), 'o-', label=result['metric'].upper())

plt.xlabel("Cluster Rank", fontsize=12)
plt.ylabel("Number of Genes", fontsize=12)
plt.title("Cluster Size Distribution by Distance Metric", fontsize=14)
plt.yscale('log')
plt.legend()
plt.grid(True, which="both", ls="--")
plt.savefig(f"{output_dir}/cluster_size_distribution.png", dpi=300, bbox_inches='tight')
plt.close()

# ----------------------------
# Save Results
# ----------------------------
for result in results:
    if result['clusters'] is not None:
        pd.DataFrame({
            'gene': top_variable_data.index,
            'cluster': result['clusters']
        }).to_csv(f"{output_dir}/clusters_{result['metric']}.csv", index=False)

# Save summary statistics
summary_data = []
for r in results:
    summary_data.append({
        'metric': r['metric'],
        'n_clusters': r.get('n_clusters', np.nan),
        'noise_ratio': r.get('noise_ratio', np.nan),
        'largest_cluster': max(r['cluster_sizes']) if 'cluster_sizes' in r and r['cluster_sizes'] else np.nan,
        'time_seconds': r.get('time', np.nan),
        'status': 'SUCCESS' if r['clusters'] is not None else 'FAILED'
    })

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(f"{output_dir}/summary_stats.csv", index=False)

print(f"\nAnalysis complete. Results saved to: {output_dir}")
print("\nSummary Statistics:")
print(summary_df.to_string(index=False))


Clustering with: EUCLIDEAN
Number of clusters: 2
Noise ratio: 4.0%
Cluster sizes: [4796, 6]
Time elapsed: 12.36 seconds

Clustering with: MANHATTAN
Number of clusters: 2
Noise ratio: 22.0%
Cluster sizes: [3896, 6]
Time elapsed: 11.52 seconds

Clustering with: CORRELATION
Number of clusters: 11
Noise ratio: 49.2%
Cluster sizes: [1273, 889, 153, 60, 56, 42, 25, 16, 13, 9, 5]
Time elapsed: 3.07 seconds

Clustering with: MAHALANOBIS
Computing Mahalanobis distances...


C:\Users\HP\AppData\Roaming\Python\Python311\site-packages\sklearn\covariance\_robust_covariance.py:753: UserWarning: The covariance matrix associated to your dataset is not full rank
  warnings.warn(
C:\Users\HP\AppData\Roaming\Python\Python311\site-packages\sklearn\covariance\_robust_covariance.py:185: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-903.793886105786214 > -918.215494810070027). You may want to try with a higher value of support_fraction (current value: 0.901).
  warnings.warn(
C:\Users\HP\AppData\Roaming\Python\Python311\site-packages\sklearn\covariance\_robust_covariance.py:185: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-903.980944325302858 > -925.641779579659897). You may want to try with a higher value of support_fraction (current value: 0.901).
  warnings.warn(
C:\Users\HP\AppData\Roaming\Python\Python311\site-packages\sklearn\covariance\_robust_covariance.py:1

Mahalanobis calculation failed: Covariance matrix is nearly singular
Using Euclidean as fallback
Number of clusters: 2
Noise ratio: 4.0%
Cluster sizes: [4796, 6]
Time elapsed: 44.68 seconds

Analysis complete. Results saved to: final_results/distance_metric_comparison_again_for_5k_GENES/check

Summary Statistics:
     metric  n_clusters  noise_ratio  largest_cluster  time_seconds  status
  euclidean           2       0.0396             4796     12.361816 SUCCESS
  manhattan           2       0.2196             3896     11.518691 SUCCESS
correlation          11       0.4918             1273      3.069968 SUCCESS
mahalanobis           2       0.0396             4796     44.681585 SUCCESS


For complete dataset 
data normalization and directory preparations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import time
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.covariance import MinCovDet
import hdbscan
from scipy import linalg

def fast_logdet(matrix):
    """Compute log determinant efficiently"""
    return 2 * np.sum(np.log(np.diag(linalg.cholesky(matrix))))

# Create output directory
output_dir = "final_results/complete_data_distance_comparison"
os.makedirs(output_dir, exist_ok=True)

# 1. Data preparation - Use complete data instead of top variable genes
normalized_data = np.log2(filtered_df + 1)  # Log2 transform with pseudocount

print(f"Complete data shape: {normalized_data.shape}")

# 2. Z-score normalization on complete data
normalized_data = pd.DataFrame(
    StandardScaler().fit_transform(normalized_data),
    index=normalized_data.index,  # Keep original gene names
    columns=normalized_data.columns
)

In [18]:
class RobustMahalanobisHDBSCAN:
    def __init__(self, min_cluster_size=5, min_samples=None):
        self.min_cluster_size = min_cluster_size
        self.min_samples = min_samples
        
    def _robust_mahalanobis(self, X):
        """Calculate pairwise Mahalanobis distances with regularization"""
        try:
            # Robust covariance estimation with increased support_fraction
            cov_estimator = MinCovDet(support_fraction=0.9).fit(X)
            cov = cov_estimator.covariance_
            
            # Regularization to ensure invertibility
            reg = 1e-6 * np.trace(cov)/cov.shape[0] * np.eye(cov.shape[0])
            cov_reg = cov + reg
            
            # Check condition number
            if fast_logdet(cov_reg) < -50:  # If nearly singular
                raise ValueError("Covariance matrix is nearly singular")
                
            cov_inv = np.linalg.inv(cov_reg)
            
            # Calculate distances using matrix operations
            diff = X[:, np.newaxis, :] - X[np.newaxis, :, :]
            dists = np.sqrt(np.einsum('...i,ij,...j->...', diff, cov_inv, diff))
            np.fill_diagonal(dists, 0)  # Ensure diagonal is exactly 0
            return dists
            
        except Exception as e:
            print(f"Mahalanobis calculation failed: {str(e)}")
            return None

    def fit(self, X):
        dist_matrix = self._robust_mahalanobis(X)
        if dist_matrix is None:
            # Fallback to Euclidean if Mahalanobis fails
            print("Using Euclidean as fallback")
            self.clusterer = hdbscan.HDBSCAN(
                min_cluster_size=self.min_cluster_size,
                min_samples=self.min_samples
            )
        else:
            self.clusterer = hdbscan.HDBSCAN(
                min_cluster_size=self.min_cluster_size,
                min_samples=self.min_samples,
                metric='precomputed'
            )
            
        self.labels_ = self.clusterer.fit_predict(dist_matrix if dist_matrix is not None else X)
        return self

# ----------------------------
# Metric Comparison on Complete Data
# ----------------------------
metrics = [
    ('euclidean', hdbscan.HDBSCAN(metric='euclidean')),
    ('manhattan', hdbscan.HDBSCAN(metric='manhattan')),
    ('correlation', hdbscan.HDBSCAN(metric='correlation')),
    ('mahalanobis', RobustMahalanobisHDBSCAN())
]

results = []
for name, clusterer in metrics:
    print(f"\n{'='*40}\nClustering with: {name.upper()}\n{'='*40}")
    
    try:
        start_time = time.time()
        
        if name == 'mahalanobis':
            print("Computing Mahalanobis distances...")
            clusterer.fit(normalized_data.values)
            clusters = clusterer.labels_
        else:
            clusters = clusterer.fit_predict(normalized_data)
            
        # Analysis
        unique_clusters = np.unique(clusters)
        n_clusters = len(unique_clusters) - 1  # Exclude noise (-1)
        noise_ratio = np.mean(clusters == -1)
        cluster_sizes = [np.sum(clusters == c) for c in unique_clusters if c != -1]
        
        results.append({
            'metric': name,
            'n_clusters': n_clusters,
            'noise_ratio': noise_ratio,
            'clusters': clusters,
            'cluster_sizes': cluster_sizes,
            'time': time.time() - start_time
        })
        
        print(f"Number of clusters: {n_clusters}")
        print(f"Noise ratio: {noise_ratio:.1%}")
        print(f"Cluster sizes: {sorted(cluster_sizes, reverse=True)}")
        print(f"Time elapsed: {results[-1]['time']:.2f} seconds")
        
    except Exception as e:
        print(f"Clustering failed: {str(e)}")
        results.append({
            'metric': name,
            'error': str(e),
            'clusters': None
        })

# ----------------------------
# Visualization
# ----------------------------
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()

# Use PCA for visualization (may take longer with complete data)
print("Computing PCA for visualization...")
pca = PCA(n_components=2).fit_transform(normalized_data)

for i, result in enumerate(results):
    ax = axes[i]
    
    if result['clusters'] is None:
        ax.text(0.5, 0.5, f"Failed:\n{result.get('error','')}", 
                ha='center', va='center', fontsize=12)
        ax.set_title(f"{result['metric'].upper()} - FAILED", fontsize=14)
        continue
        
    sc = ax.scatter(
        pca[:, 0], pca[:, 1],
        c=result['clusters'],
        cmap='tab20',
        alpha=0.7,
        s=15
    )
    
    title = (
        f"{result['metric'].upper()}\n"
        f"Clusters: {result['n_clusters']} | "
        f"Noise: {result['noise_ratio']:.1%}\n"
        f"Time: {result['time']:.2f}s"
    )
    ax.set_title(title, fontsize=12)
    
    # Add colorbar for cluster IDs
    plt.colorbar(sc, ax=ax, label='Cluster ID')

plt.tight_layout()
plt.savefig(f"{output_dir}/complete_data_metric_comparison.png", dpi=300, bbox_inches='tight')
plt.close()

# ----------------------------
# Cluster Size Distribution Plot
# ----------------------------
plt.figure(figsize=(12, 6))
for result in results:
    if result['clusters'] is not None and result['cluster_sizes']:
        sizes = result['cluster_sizes']
        plt.plot(sorted(sizes, reverse=True), 'o-', label=result['metric'].upper())

plt.xlabel("Cluster Rank", fontsize=12)
plt.ylabel("Number of Genes", fontsize=12)
plt.title("Cluster Size Distribution by Distance Metric (Complete Data)", fontsize=14)
plt.yscale('log')
plt.legend()
plt.grid(True, which="both", ls="--")
plt.savefig(f"{output_dir}/complete_data_cluster_size_distribution.png", dpi=300, bbox_inches='tight')
plt.close()

# ----------------------------
# Save Results
# ----------------------------
for result in results:
    if result['clusters'] is not None:
        pd.DataFrame({
            'gene': normalized_data.index,
            'cluster': result['clusters']
        }).to_csv(f"{output_dir}/complete_data_clusters_{result['metric']}.csv", index=False)

# Save summary statistics
summary_data = []
for r in results:
    summary_data.append({
        'metric': r['metric'],
        'n_clusters': r.get('n_clusters', np.nan),
        'noise_ratio': r.get('noise_ratio', np.nan),
        'largest_cluster': max(r['cluster_sizes']) if 'cluster_sizes' in r and r['cluster_sizes'] else np.nan,
        'time_seconds': r.get('time', np.nan),
        'status': 'SUCCESS' if r['clusters'] is not None else 'FAILED'
    })

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(f"{output_dir}/complete_data_summary_stats.csv", index=False)

print(f"\nAnalysis complete. Results saved to: {output_dir}")
print("\nSummary Statistics:")
print(summary_df.to_string(index=False))

Complete data shape: (27038, 202)

Clustering with: EUCLIDEAN


/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Number of clusters: 63
Noise ratio: 58.3%
Cluster sizes: [np.int64(10357), np.int64(131), np.int64(48), np.int64(43), np.int64(36), np.int64(30), np.int64(30), np.int64(27), np.int64(27), np.int64(25), np.int64(24), np.int64(22), np.int64(19), np.int64(19), np.int64(17), np.int64(17), np.int64(15), np.int64(14), np.int64(14), np.int64(13), np.int64(13), np.int64(13), np.int64(12), np.int64(12), np.int64(12), np.int64(12), np.int64(12), np.int64(11), np.int64(11), np.int64(11), np.int64(10), np.int64(10), np.int64(10), np.int64(10), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5)]
Time elapsed: 87.07 seconds

Clustering with: MANHATTAN


/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Number of clusters: 62
Noise ratio: 60.0%
Cluster sizes: [np.int64(9945), np.int64(116), np.int64(48), np.int64(43), np.int64(35), np.int64(34), np.int64(30), np.int64(27), np.int64(26), np.int64(22), np.int64(20), np.int64(17), np.int64(17), np.int64(17), np.int64(16), np.int64(15), np.int64(14), np.int64(14), np.int64(14), np.int64(13), np.int64(13), np.int64(12), np.int64(12), np.int64(12), np.int64(12), np.int64(12), np.int64(11), np.int64(11), np.int64(10), np.int64(10), np.int64(10), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(5), np.int64(5), np.int64(5), np.int64(5)]
Time elapsed: 94.60 seconds

Clustering with: CORRELATION


/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Number of clusters: 76
Noise ratio: 86.7%
Cluster sizes: [np.int64(1929), np.int64(184), np.int64(156), np.int64(135), np.int64(131), np.int64(62), np.int64(58), np.int64(51), np.int64(50), np.int64(48), np.int64(43), np.int64(34), np.int64(33), np.int64(27), np.int64(27), np.int64(25), np.int64(25), np.int64(24), np.int64(20), np.int64(18), np.int64(17), np.int64(17), np.int64(17), np.int64(17), np.int64(15), np.int64(14), np.int64(14), np.int64(14), np.int64(13), np.int64(13), np.int64(13), np.int64(13), np.int64(13), np.int64(12), np.int64(12), np.int64(12), np.int64(12), np.int64(11), np.int64(11), np.int64(10), np.int64(10), np.int64(10), np.int64(10), np.int64(10), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(8), np.int64(8), np.int64(8), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(5), np.int64

/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/covariance/_robust_covariance.py:749: UserWarning: The covariance matrix associated to your dataset is not full rank
  warnings.warn(
/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/covariance/_robust_covariance.py:185: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-1146.262461109045489 > -1148.793936256348161). You may want to try with a higher value of support_fraction (current value: 0.900).
  warnings.warn(
/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/covariance/_robust_covariance.py:185: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-1072.265073565933790 > -1076.612386363824044). You may want to try with a higher value of support_fraction (current value: 0.900).
  warnings.warn(
/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/covariance/_robust_covariance.py:185: 

Mahalanobis calculation failed: Covariance matrix is nearly singular
Using Euclidean as fallback
Number of clusters: 63
Noise ratio: 58.3%
Cluster sizes: [np.int64(10357), np.int64(131), np.int64(48), np.int64(43), np.int64(36), np.int64(30), np.int64(30), np.int64(27), np.int64(27), np.int64(25), np.int64(24), np.int64(22), np.int64(19), np.int64(19), np.int64(17), np.int64(17), np.int64(15), np.int64(14), np.int64(14), np.int64(13), np.int64(13), np.int64(13), np.int64(12), np.int64(12), np.int64(12), np.int64(12), np.int64(12), np.int64(11), np.int64(11), np.int64(11), np.int64(10), np.int64(10), np.int64(10), np.int64(10), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), n